# OpenCV Label / Template Matching on a Worksheet

Load one worksheet image (downloaded from its remote `image_url`), pick a single
**symbol** as a template, and find every similar patch on the sheet using
multi-scale `cv2.matchTemplate` + non-maximum suppression.

**Flow**
1. Pick a request (`rid`) and worksheet (`wid`) from `data/requests/`.
2. Download the worksheet raster from its `image_url`.
3. Crop a template patch (one symbol).
4. Multi-scale normalized cross-correlation matching.
5. NMS to dedupe + visualize all hits.

In [ ]:
import io
import json
from pathlib import Path

import cv2
import numpy as np
import requests
import matplotlib.pyplot as plt
from PIL import Image

# Resolve project root from this notebook (eda/notebooks/ -> project root)
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data" / "requests").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
REQUESTS_DIR = PROJECT_ROOT / "data" / "requests"
print("Project root:", PROJECT_ROOT)
print("Requests dir exists:", REQUESTS_DIR.exists())


def list_requests():
    return sorted(
        p.name for p in REQUESTS_DIR.iterdir()
        if p.is_dir() and (p / "worksheets_metadata.json").exists()
    )


def load_worksheets(rid):
    meta = json.load(open(REQUESTS_DIR / rid / "worksheets_metadata.json"))
    return [
        {
            "wid": w["id"],
            "name": w.get("name"),
            "title": w.get("title"),
            "url": (w.get("image") or {}).get("image_url"),
            "width": (w.get("image") or {}).get("width"),
            "height": (w.get("image") or {}).get("height"),
        }
        for w in meta
        if (w.get("image") or {}).get("image_url")
    ]


def download_image(url):
    """Download an image_url into a BGR uint8 numpy array."""
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    pil = Image.open(io.BytesIO(resp.content)).convert("RGB")
    return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)

## 1. Pick a request and worksheet

In [ ]:
rids = list_requests()
print(f"{len(rids)} cached requests. First few:")
for r in rids[:10]:
    print("  ", r)

# Choose a request id (override RID with any value printed above)
RID = rids[0]
worksheets = load_worksheets(RID)
print(f"\nRID = {RID}\n{len(worksheets)} worksheets with images. First few:")
for w in worksheets[:10]:
    print(f"  {w['wid']}  {w['name']!s:10}  {w['title']}")

In [ ]:
# Choose a worksheet (override WID with any wid printed above)
WID = worksheets[0]["wid"]
ws = next(w for w in worksheets if w["wid"] == WID)
print("Loading worksheet:", ws["name"], "—", ws["title"])
print("image_url:", ws["url"])

img = download_image(ws["url"])      # BGR
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
print("Image shape (H, W, C):", img.shape)

plt.figure(figsize=(16, 11))
plt.imshow(img_rgb)
plt.title(f"{ws['name']} — {ws['title']}  ({img.shape[1]}x{img.shape[0]})")
plt.axis("off")
plt.show()

## 2. Pick a template patch (one symbol)

Set the bounding box `(x, y, w, h)` in **image pixels** around a single symbol.
Zoom into the image above (or use the preview below) to find good coordinates.

In [ ]:
# Template bounding box in image pixels — EDIT THESE to frame one symbol.
TPL_X, TPL_Y, TPL_W, TPL_H = 3300, 2400, 90, 90

template = img[TPL_Y:TPL_Y + TPL_H, TPL_X:TPL_X + TPL_W]
print("Template shape:", template.shape)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(cv2.cvtColor(template, cv2.COLOR_BGR2RGB))
ax[0].set_title("Selected template")
ax[0].axis("off")

# Context: where the template sits on the sheet
ctx = img_rgb.copy()
cv2.rectangle(ctx, (TPL_X, TPL_Y), (TPL_X + TPL_W, TPL_Y + TPL_H), (255, 204, 51), 8)
ax[1].imshow(ctx)
ax[1].set_title("Template location")
ax[1].axis("off")
plt.show()

## 3. Multi-scale template matching + NMS

In [ ]:
def nms(boxes, iou_thresh=0.3):
    """Greedy non-maximum suppression. boxes = list of (x, y, w, h, score)."""
    if not boxes:
        return []
    boxes = sorted(boxes, key=lambda b: b[4], reverse=True)
    keep = []
    for cand in boxes:
        cx1, cy1, cx2, cy2 = cand[0], cand[1], cand[0] + cand[2], cand[1] + cand[3]
        cand_area = cand[2] * cand[3]
        drop = False
        for k in keep:
            kx1, ky1, kx2, ky2 = k[0], k[1], k[0] + k[2], k[1] + k[3]
            ix1, iy1 = max(cx1, kx1), max(cy1, ky1)
            ix2, iy2 = min(cx2, kx2), min(cy2, ky2)
            iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
            inter = iw * ih
            if inter == 0:
                continue
            union = cand_area + k[2] * k[3] - inter
            if union > 0 and inter / union > iou_thresh:
                drop = True
                break
        if not drop:
            keep.append(cand)
    return keep


def match_symbol(image, template, threshold=0.7, scales=(0.8, 0.9, 1.0, 1.1, 1.25),
                 iou_thresh=0.3, max_raw=4000):
    """Find all patches in `image` similar to `template` via multi-scale TM_CCOEFF_NORMED."""
    img_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    tpl_gray = cv2.cvtColor(template, cv2.COLOR_BGR2GRAY)
    th0, tw0 = tpl_gray.shape[:2]

    raw = []
    for s in scales:
        tw, th = max(4, int(round(tw0 * s))), max(4, int(round(th0 * s)))
        if th >= img_gray.shape[0] or tw >= img_gray.shape[1]:
            continue
        tpl = cv2.resize(tpl_gray, (tw, th), interpolation=cv2.INTER_AREA)
        res = cv2.matchTemplate(img_gray, tpl, cv2.TM_CCOEFF_NORMED)
        ys, xs = np.where(res >= threshold)
        for x, y in zip(xs.tolist(), ys.tolist()):
            raw.append((int(x), int(y), tw, th, float(res[y, x])))
        if len(raw) > max_raw:
            raw = sorted(raw, key=lambda b: b[4], reverse=True)[:max_raw]

    return nms(raw, iou_thresh=iou_thresh)


THRESHOLD = 0.70
matches = match_symbol(img, template, threshold=THRESHOLD)
print(f"Found {len(matches)} matches at threshold {THRESHOLD}")

## 4. Visualize all matches

In [ ]:
canvas = img_rgb.copy()
for (x, y, w, h, score) in matches:
    cv2.rectangle(canvas, (x, y), (x + w, y + h), (46, 204, 113), 6)   # green = match
# template box on top (yellow)
cv2.rectangle(canvas, (TPL_X, TPL_Y), (TPL_X + TPL_W, TPL_Y + TPL_H), (255, 204, 51), 8)

plt.figure(figsize=(18, 12))
plt.imshow(canvas)
plt.title(f"{len(matches)} matches  |  threshold={THRESHOLD}  |  {ws['name']} — {ws['title']}")
plt.axis("off")
plt.show()

In [ ]:
# Optional: sweep the threshold to help you tune it for this symbol.
for t in [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]:
    n = len(match_symbol(img, template, threshold=t))
    print(f"threshold {t:.2f} -> {n} matches")

## 5. FastSAM box-finding from reference points

Use the worksheet's annotated **reference points** to segment each symbol into a
bounding box. MobileSAM's single-point masks were too loose on these schematics,
so we use **FastSAM** instead, run on a small crop around each point (the sheets
are ~7000×5000 with tiny symbols), keeping the smallest mask that covers the
point. A dark connected-component box is used as a fallback when FastSAM misses.

The reference points come from `worksheet_geometries/<wid>_geometries.json`,
filtered the same way as the EDA notebooks — keep `feature.geometry_type == 1`
(Point) outputs ("electrical filtering"), then map each Feathers coordinate
`[x, y]` (y is negative) to image pixels `(x, -y)`.

This reuses the app modules `worksheet_loader`, `sam_boxes` (point loader) and
`fastsam_boxes` so the notebook and the FastAPI app share one implementation.
We also pick a sheet that actually **has geometries**. Requires
`models/FastSAM-s.pt`.

In [ ]:
import sys
import glob
import os

# Reuse the app's modules (single source of truth for SAM + loading).
APP_DIR = PROJECT_ROOT / "symbol_matcher_app"
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

import worksheet_loader as wl
import sam_boxes as sb          # reference-point loader (electrical filter)
import fastsam_boxes as fsb     # FastSAM crop-wise box finder


def first_worksheet_with_points(rid, min_points=8):
    """Find a worksheet in `rid` that has cached point references + an image."""
    meta = {w["id"]: w for w in load_worksheets(rid)}  # only image-bearing wids
    gdir = REQUESTS_DIR / rid / "worksheet_geometries"
    for gp in sorted(glob.glob(str(gdir / "*_geometries.json"))):
        wid = os.path.basename(gp).replace("_geometries.json", "")
        if wid not in meta:
            continue
        w, h = meta[wid]["width"], meta[wid]["height"]
        if len(sb.load_reference_points(rid, wid, w, h)) >= min_points:
            return wid
    return None


# Pick a worksheet that actually has reference points (override SAM_WID freely).
SAM_RID = RID
SAM_WID = first_worksheet_with_points(SAM_RID) or WID
_meta = {w["wid"]: w for w in load_worksheets(SAM_RID)}.get(SAM_WID, {})
print("SAM worksheet:", SAM_RID, SAM_WID, "| page", _meta.get("page_no"), "-", _meta.get("name"))

sam_img = wl.load_worksheet_image(SAM_RID, SAM_WID)        # BGR
sam_img_rgb = cv2.cvtColor(sam_img, cv2.COLOR_BGR2RGB)
H, W = sam_img.shape[:2]
ref_points = sb.load_reference_points(SAM_RID, SAM_WID, W, H)
print(f"{len(ref_points)} reference points (geometry_type == 1) on a {W}x{H} sheet")

In [ ]:
# FastSAM on a small crop around each reference point -> tight box per symbol.
# (First call loads the FastSAM model.)
import time

LIMIT = 60  # cap points for a quick demo; set None to use all reference points
pts = ref_points[:LIMIT] if LIMIT else ref_points

t = time.time()
sam_boxes_found = fsb.boxes_from_points(sam_img, pts)
print(f"{len(sam_boxes_found)} boxes from {len(pts)} points in {time.time() - t:.1f}s")
for b in sam_boxes_found[:5]:
    print(b.as_dict())

In [ ]:
# Visualize: reference points (red) and MobileSAM boxes (cyan).
canvas = sam_img_rgb.copy()
for p in pts:
    cv2.circle(canvas, (p.x, p.y), 5, (255, 0, 0), -1)
for b in sam_boxes_found:
    cv2.rectangle(canvas, (b.x, b.y), (b.x + b.w, b.y + b.h), (0, 229, 255), 4)

plt.figure(figsize=(18, 12))
plt.imshow(canvas)
plt.title(f"MobileSAM: {len(sam_boxes_found)} boxes from {len(pts)} reference points")
plt.axis("off")
plt.show()

## 6. HQ-SAM box-finding (crisper on thin symbols)

FastSAM is fast but can be loose in dense regions. **Light HQ-SAM** (`vit_tiny`
backbone + high-quality mask head) gives sharper masks on thin line-drawing
symbols. We run it crop-wise like FastSAM, but additionally feed the neighbouring
reference points as **negative** prompts so the mask stops at the gap between
adjacent symbols. Shared with the app via `hqsam_boxes`.

Weights: `models/sam_hq_vit_tiny.pth` (from `huggingface.co/lkeab/hq-sam`).

In [ ]:
import hqsam_boxes as hsb     # Light HQ-SAM crop-wise box finder (shares point loader)

# Same reference points as the FastSAM demo above. First call loads HQ-SAM.
t = time.time()
hq_boxes = hsb.boxes_from_points(sam_img, pts)
print(f"HQ-SAM: {len(hq_boxes)} boxes from {len(pts)} points in {time.time() - t:.1f}s")

# Visualize: reference points (red) and HQ-SAM boxes (magenta).
canvas_hq = sam_img_rgb.copy()
for p in pts:
    cv2.circle(canvas_hq, (p.x, p.y), 5, (255, 0, 0), -1)
for b in hq_boxes:
    cv2.rectangle(canvas_hq, (b.x, b.y), (b.x + b.w, b.y + b.h), (255, 61, 240), 4)

plt.figure(figsize=(18, 12))
plt.imshow(canvas_hq)
plt.title(f"HQ-SAM: {len(hq_boxes)} boxes from {len(pts)} reference points")
plt.axis("off")
plt.show()

## 7. Multi-scale glow pseudo-coloring (SAM preprocessing)

SAM-family models are trained on natural photos and struggle with razor-thin,
1-px monochrome CAD lines. `pseudocolor.pseudo_color(..., invert=True)` ("glow"
mode) blurs each channel with a different Gaussian kernel — red `3x3` (sharp),
green `7x7` (context), blue `13x13` (wide glow) — producing a coloured
chromatic-aberration gradient around every line that SAM perceives like natural
depth/lighting.

Below: a raw crop vs its glow version, then FastSAM run **without** and **with**
pseudo-coloring (`pseudocolor=True`) on the same points. In the app this is the
**Pseudo-color** checkbox / `?pseudocolor=1`. Only the model *input* is
pseudo-colored; box coordinates and the connected-component fallback stay on the
original image.

In [ ]:
import pseudocolor as pc     # multi-scale glow pseudo-coloring (shared with the app)

# Glow the full sheet once, then show a dense crop raw vs pseudo-colored.
glow_img = pc.pseudo_color(sam_img, invert=True)          # BGR
cxp, cyp = (ref_points[0].x, ref_points[0].y) if ref_points else (W // 2, H // 2)
R = 180
gx0, gy0 = max(0, cxp - R), max(0, cyp - R)
gx1, gy1 = min(W, cxp + R), min(H, cyp + R)

fig, ax = plt.subplots(1, 2, figsize=(14, 7))
ax[0].imshow(sam_img_rgb[gy0:gy1, gx0:gx1]); ax[0].set_title("raw crop"); ax[0].axis("off")
ax[1].imshow(cv2.cvtColor(glow_img[gy0:gy1, gx0:gx1], cv2.COLOR_BGR2RGB))
ax[1].set_title("glow pseudo-color (invert=True)"); ax[1].axis("off")
plt.show()

# FastSAM without vs with pseudo-coloring on the same points.
t = time.time()
boxes_off = fsb.boxes_from_points(sam_img, pts, pseudocolor=False)
boxes_on = fsb.boxes_from_points(sam_img, pts, pseudocolor=True)
print(f"FastSAM  off: {len(boxes_off)} boxes | pseudocolor: {len(boxes_on)} boxes "
      f"({time.time() - t:.1f}s total)")

fig, ax = plt.subplots(1, 2, figsize=(20, 12))
for axi, boxes, title in ((ax[0], boxes_off, "FastSAM (raw)"),
                          (ax[1], boxes_on, "FastSAM (pseudo-color)")):
    canvas = sam_img_rgb.copy()
    for p in pts:
        cv2.circle(canvas, (p.x, p.y), 5, (255, 0, 0), -1)
    for b in boxes:
        cv2.rectangle(canvas, (b.x, b.y), (b.x + b.w, b.y + b.h), (0, 200, 255), 4)
    axi.imshow(canvas); axi.set_title(f"{title}: {len(boxes)} boxes"); axi.axis("off")
plt.show()